# Predict volcano eruption
**Volcanic earthquakes** have various characteristics depending on the volcano, and there are various classifications, but the typical classifications can be classified as follows.

**Volcanic earthquake**
 * type A (10Hz or higher): Earthquake caused by destruction of rocks surrounding magma chambers and conduits
 * type BH (5-8Hz) : Earthquake caused by magma intruding into the conduit and destroying the conduit and rocks around the conduit
 * type BL (1.5-2.5Hz) : An earthquake around the conduit due to gas etc. ejecting from the crater prior to the explosive eruption and reducing the pressure inside the conduit.

**Volcanic tremor**
 * type C (0.5-1.2Hz) : Vibration due to increase in gas pressure in the cavity along with BH
 * type D (2-4Hz) : Vibration due to gas ejection along with BL

The feature and model code lives in the `ingv_eruption` package of this repository; this notebook calls it and draws the charts.

In [ ]:
### Kaggle or Local-PC ###
KAGGLE = True       # <==== SET ============
BRANCH = 'main'

if KAGGLE:          # needs "Internet" switched on in the notebook settings
    DIR = '../input/predict-volcanic-eruptions-ingv-oe'
    REPO = 'repo'
    !git clone --depth 1 --branch {BRANCH} https://github.com/TNguyen1228/LANL-Earthquake-Prediction.git {REPO}
    !pip install -q ./{REPO}
else:               # local PC, started from the notebooks/ folder after `pip install -e ..`
    DIR = './predict-volcanic-eruptions-ingv-oe/'
    REPO = '..'

In [ ]:
import os
import datetime

import numpy as np
import pandas as pd
import scipy
import scipy.signal

import matplotlib
import matplotlib.pyplot as plt
pd.options.display.max_columns = None    # disp all columns
%matplotlib inline

from sklearn.model_selection import train_test_split

from ingv_eruption import build_feature_table, train_model, predict

# Data

In [ ]:
train = pd.read_csv(os.path.join(DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DIR, 'sample_submission.csv'))

train

In [ ]:
# Convert 'time_to_eruption'to hours:minutes:seconds (Just for reference)
train['h:m:s'] = (train['time_to_eruption']
                  .apply(lambda x:datetime.timedelta(seconds = x/100)))
train

# Observe sample data

In [ ]:
# Select sample segment
sample_df = (train.sort_values('time_to_eruption')
             .reset_index()
             .rename(columns={'index': 'train_id'}))
sample_df = sample_df[sample_df.index % (len(train) // 5) == 5].reset_index(drop = True)
sample_ids = sample_df['segment_id'].values
sample_df

### Time Domain

In [ ]:
sensor = 3     #### 1 ~ 10

fig, ax = plt.subplots(len(sample_ids), 1, figsize = (12, len(sample_ids)*2))
for i, segment_id in enumerate(sample_ids):
    segment_df = pd.read_csv(os.path.join(DIR, f'train/{segment_id}.csv')).fillna(0)
    ax[i].plot(range(len(segment_df)), segment_df[f'sensor_{sensor}'])
    ax[i].set_title(f'segment_id : {segment_id},  sensor : {sensor}')

fig.tight_layout()

### Time-Frequency Domain (STFT)
STFT : Short Time Fourier Transform

In [ ]:
fs = 100                # sampling frequency
n = 256                 # FFT segment size

fig, ax = plt.subplots(len(sample_ids), 1, figsize = (12, len(sample_ids)*2))
for i, segment_id in enumerate(sample_ids):
    segment_df = pd.read_csv(os.path.join(DIR, f'train/{segment_id}.csv')).fillna(0)

    x = segment_df[f'sensor_{sensor}']
    f, t, Z = scipy.signal.stft(x, fs = fs, window = 'hann', nperseg = n)
    Z = np.abs(Z)

    ax[i].pcolormesh(t, f, Z, vmin = 0, vmax = Z.mean()*10)
    ax[i].set_ylim(0, 20)
    ax[i].set_ylabel('Frequency [Hz]'); plt.xlabel('Time [s]')
    ax[i].set_title(f'segment_id : {segment_id},  sensor : {sensor}')
fig.tight_layout()

# Features

In [ ]:
feature_df = build_feature_table(DIR, 'train', train['segment_id'])
train_set = pd.merge(train, feature_df, on = 'segment_id')
train_set

### Check against the 2023 run
`tests/reference/features_2023.csv` holds the feature rows that the original notebook displayed in December 2023 (printed with six decimals). On the real data set every compared row should match.

In [ ]:
def check_against_2023(feature_df, split):
    ref = pd.read_csv(os.path.join(REPO, 'tests/reference/features_2023.csv'))
    ref = ref[ref['split'] == split].drop(columns = 'split').set_index('segment_id')
    new = feature_df.set_index('segment_id')
    common = ref.index.intersection(new.index)
    if len(common) == 0:
        print(f'{split}: no reference segment in this data set, nothing compared')
        return
    a, b = new.loc[common, ref.columns].to_numpy(float), ref.loc[common].to_numpy(float)
    same = np.isclose(a, b, rtol = 1e-6, atol = 1e-5, equal_nan = True)
    print(f'{split}: {len(common)} segments compared, {same.sum()} of {same.size} values match')
    assert same.all(), f'{(~same).sum()} feature values differ from the 2023 run'

check_against_2023(feature_df, 'train')

In [ ]:
fig, ax = plt.subplots(2, 5, figsize = (12, 6))
x = train_set['time_to_eruption']
for i,type in enumerate(['A_pow','A_num','BH_pow','BH_num','BL_pow','BL_num','C_pow','C_num','D_pow','D_num']):
    y = np.zeros(len(x))
    for j in range(10):
        y += train_set[f's{j+1}_{type}']
    y /= 10
    x1 = np.polyfit(x, y.fillna(y.mean()), 2)
    y1 = np.poly1d(x1)(x)
    ax[i%2, i//2].plot(x, y,'.')
    ax[i%2, i//2].plot(x, y1,'.')
    ax[i%2, i//2].set_ylim(0,)
    ax[i%2, i//2].set_title(type)
fig.tight_layout()

# Modeling and Predicting

In [ ]:
df = train_set.drop(['segment_id', 'time_to_eruption','h:m:s'], axis=1)
y = train_set['time_to_eruption']

X_train, X_val, y_train, y_val = train_test_split(df, y,
                                                  random_state = 42,
                                                  test_size = 0.2,
                                                  shuffle = True)

In [ ]:
lgb_model = train_model(X_train, y_train, X_val, y_val)

# Predict test data

In [ ]:
feature_df = build_feature_table(DIR, 'test', test['segment_id'])
check_against_2023(feature_df, 'test')

In [ ]:
# Predict test data
result = predict(lgb_model, feature_df)
result['h:m:s']=(result['time_to_eruption'].apply(lambda x:datetime.timedelta(seconds = x/100)))
result

In [ ]:
result[['segment_id','time_to_eruption']].to_csv('submission.csv', index=False)